## 문자 단위 RNN(Char RNN)

In [54]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


In [55]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print ('문자 집합의 크기 : {}'.format(vocab_size))


문자 집합의 크기 : 5


In [56]:
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1


In [57]:
char_to_index = dict((c, i) for i, c in enumerate(char_vocab)) # 문자에 고유한 정수 인덱스 부여
print(char_to_index)


{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


In [58]:
index_to_char={}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)


{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [59]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)


[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [60]:
# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음.
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)


[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [61]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)


[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [62]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)


In [63]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))


훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 5])


In [64]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) # RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) # 출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x


In [65]:
net = Net(input_size, hidden_size, output_size)


In [66]:
outputs = net(X)
print(outputs.shape) # 3차원 텐서


torch.Size([1, 5, 5])


In [67]:
torch.Size([1, 5, 5])


torch.Size([1, 5, 5])

In [68]:
print(outputs.view(-1, input_size).shape) # 2차원 텐서로 변환


torch.Size([5, 5])


In [69]:
torch.Size([5, 5])


torch.Size([5, 5])

In [70]:
print(Y.shape)
print(Y.view(-1).shape)


torch.Size([1, 5])
torch.Size([5])


In [71]:
torch.Size([1, 5])
torch.Size([5])


torch.Size([5])

In [72]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)


In [73]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view를 하는 이유는 Batch 차원 제거를 위해
    loss.backward() # 기울기 계산
    optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드.
    result = outputs.data.numpy().argmax(axis=2) # 최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)


0 loss:  1.8306283950805664 prediction:  [[1 0 1 1 1]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  a!aaa
1 loss:  1.5463756322860718 prediction:  [[0 0 0 0 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  !!!!!
2 loss:  1.3693821430206299 prediction:  [[0 3 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  !lle!
3 loss:  1.1925146579742432 prediction:  [[4 3 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  plle!
4 loss:  1.0115032196044922 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
5 loss:  0.8349508047103882 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
6 loss:  0.680456280708313 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
7 loss:  0.5622488260269165 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
8 loss:  0.4745400846004486 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
9 loss:  0.404951274394989 prediction:  [[4 4 4 2 0]] tr

In [74]:
import torch
import torch.nn as nn
import torch.optim as optim


In [75]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")


In [76]:
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩


In [77]:
print(char_dic) # 공백도 여기서는 하나의 원소


{' ': 0, 'o': 1, 'l': 2, 'k': 3, 'a': 4, 'g': 5, "'": 6, 'h': 7, '.': 8, 'b': 9, 'c': 10, 'm': 11, 's': 12, 'r': 13, 'p': 14, 't': 15, 'y': 16, 'n': 17, 'f': 18, 'u': 19, 'w': 20, 'i': 21, 'e': 22, 'd': 23, ',': 24}


In [78]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))


문자 집합의 크기 : 25


In [79]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1


In [80]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index


0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [81]:
print(x_data[0])
print(y_data[0])


[21, 18, 0, 16, 1, 19, 0, 20, 4, 17]
[18, 0, 16, 1, 19, 0, 20, 4, 17, 15]


In [82]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)


In [83]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))


훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [84]:
print(X[0])


tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,

In [85]:
print(Y[0])


tensor([18,  0, 16,  1, 19,  0, 20,  4, 17, 15])


In [86]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x


In [87]:
net = Net(dic_size, hidden_size, 2) # 이번에는 층을 두 개 쌓습니다.


In [88]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)


In [89]:
outputs = net(X)
print(outputs.shape) # 3차원 텐서


torch.Size([170, 10, 25])


In [90]:
torch.Size([170, 10, 25])


torch.Size([170, 10, 25])

In [91]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환.


torch.Size([1700, 25])


In [92]:
torch.Size([1700, 25])


torch.Size([1700, 25])

In [93]:
print(Y.shape)
print(Y.view(-1).shape)


torch.Size([170, 10])
torch.Size([1700])


In [94]:
torch.Size([170, 10])
torch.Size([1700])


torch.Size([1700])

In [95]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)


pwpwg.gwpwwp.p.pwgwgppgtw.pwg.pp.p..p.wwpw.pwpw.wggp.pgwwpggp..p.wg.wpwpg.ppp.p.pp.p.pwtpppgpwwp.gpwwtppp.p.ppwpgpwgwpwgwwpggwpww.pwwp.gp..gp.pwpwggwwp.p..ggwp.pwwpwpgwg..pgwwptpg
yoooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooootoooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
eeee e e ee e e e  e e e  e e   e eee e e e e  eeeee e e   e   ee   e  e    ee e  e e e e eeee e e eeee e e e e   e e      ee   e    e  eeeee e e  e e  e  eee    e  ee eee  eeeee 
 mbb mb  mr r'mmmrmr  mmrmymrmrmmrrmy rm'  mry rmm mmymrm'  mym rmy rmmymmm'  m'rmrmrmr rmy rmmmrr''  rmr  mrmrmymrmrmmmrmb  m'  rmr rmrm'  ymr rmmmymy rmr  rr'' r'm rrm rrmr   rm
  rm  rml t         tt         tt        o o tt  t       t   mot  lo   t   mtt       mt    t  mo    o  t         t   mt   n   mt   l     mt           t   lm    o   mt  tot   mt   
          t                     t              t     t                       t                      